# LlamaIndex + Llama.cpp

## Configuration

In [ ]:
#https://docs.llamaindex.ai/en/stable/understanding/rag/
#https://docs.llamaindex.ai/en/stable/examples/low_level/oss_ingestion_retrieval/

In [ ]:
!pip install llama-index-readers-file pymupdf

In [ ]:
!pip install llama-index-vector-stores-chroma

In [ ]:
!pip install llama-index-embeddings-huggingface

In [ ]:
!pip install llama-index-llms-llama-cpp

In [ ]:
# !pip install opencv-python-headless==4.8.0.74

In [ ]:
!pip install llama-cpp-python

In [ ]:
!pip show llama-index

## Model

**llama-2-chat-13b-ggml model**

In [ ]:
from llama_index.llms.llama_cpp import LlamaCPP

# model_url = "https://huggingface.co/TheBloke/Llama-2-13B-chat-GGML/resolve/main/llama-2-13b-chat.ggmlv3.q4_0.bin"
model_url = "https://huggingface.co/TheBloke/Llama-2-13B-chat-GGUF/resolve/main/llama-2-13b-chat.Q4_0.gguf"

llm = LlamaCPP(
    # You can pass in the URL to a GGML model to download it automatically
    model_url=model_url,
    # optionally, you can set the path to a pre-downloaded model instead of model_url
    model_path=None,
    temperature=0,
    max_new_tokens=256,
    # llama2 has a context window of 4096 tokens, but we set it lower to allow for some wiggle room
    context_window=4096,
    # kwargs to pass to __call__()
    generate_kwargs={},
    # kwargs to pass to __init__()
    # set to at least 1 to use GPU
    model_kwargs={"n_gpu_layers": 1},
    verbose=True,
)

In [ ]:
# sentence transformers
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en") 

#### Chromadb configuration

**EphemeralClient** is suitable for in-memory storage and does not persist data to disk.

To save data to disk using DuckDB/Parquet, we need to use **PersistentClient** or configure the persist_directory in the settings properly.

In [ ]:
# from llama_index.vector_stores.chroma import ChromaVectorStore
# from chromadb.config import Settings
# import chromadb

# # create client and a new collection
# chroma_client = chromadb.EphemeralClient()
# chroma_collection = chroma_client.create_collection("Collection1")


# # Initialize the Chroma vector store
# vector_store = ChromaVectorStore(
#     persist_directory="./chroma_data",  # Directory to save ChromaDB data
#     chroma_collection=chroma_collection,
#     embedding_dim=384  # Dimension of OpenAI embeddings
# )

**OR**

In [ ]:
from llama_index.vector_stores.chroma import ChromaVectorStore
from chromadb import Client
from chromadb.config import Settings
import chromadb

#before  https://docs.trychroma.com/production/administration/migration
# Configure ChromaDB for persistent storage
# chroma_settings = Settings(
#     chroma_db_impl="duckdb+parquet",
#     persist_directory="./chroma_data"  # Directory to save ChromaDB data
# )
# Create a Persistent Client
# chroma_client = Client(settings=chroma_settings)

#after
chroma_client = chromadb.PersistentClient(path="./chroma_data") 

# Create or get a collection
chroma_collection1 = chroma_client.get_or_create_collection("Collection3")

# Initialize the Chroma vector store
vector_store2 = ChromaVectorStore(
    chroma_collection=chroma_collection1,
    persist_directory="./chroma_data",  # Directory to save ChromaDB data
    embedding_dim=384  # Dimension of OpenAI embeddings
)


In [ ]:
node_count = chroma_collection1.count()
print(f"Number of nodes: {node_count}")

## Data Indexing

### Loading and splitting data

In [ ]:
from pathlib import Path
from llama_index.readers.file import PyMuPDFReader
from llama_index.core.node_parser import SentenceSplitter


# Define file paths
file_paths = [
    "./OPEN SOURCE SOFTWARE GUIDELINES.pdf",
    "./Creating Impactful.pdf"
]

# Initialize loader and splitter
loader = PyMuPDFReader()
text_parser = SentenceSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separator=" "  # Define your separator if needed
)

# Load documents and process 
text_chunks = []
doc_idxs = []

for file_idx, file_path in enumerate(file_paths):
    try:
        documents = loader.load(file_path=file_path)
        for doc_idx, doc in enumerate(documents):
            cur_text_chunks = text_parser.split_text(doc.text)
            text_chunks.extend(cur_text_chunks)
            doc_idxs.extend([file_idx] * len(cur_text_chunks))  # Track file index
    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except Exception as e:
        print(f"An error occurred while loading {file_path}: {e}")

# Debug: Output results
print(f"Total number of chunks: {len(text_chunks)}")
if text_chunks:
    print(f"First chunk from {file_paths[0]}: {text_chunks[0]}")
    print(f"First chunk from {file_paths[1]}: {text_chunks[len(text_chunks) // 2]}")


### Manually Construct Nodes from Text Chunks

In [ ]:
from llama_index.core.schema import TextNode

nodes = []
for idx, doc in enumerate(text_chunks):
    # Extract text content from the Document instance
    text_content = doc.page_content if hasattr(doc, 'page_content') else str(doc)
    node = TextNode(
        text=text_content,  # Ensure this is a string
    )
    nodes.append(node)


###  Generate Embeddings for each Node

In [ ]:
for node in nodes:
    node_embedding = embed_model.get_text_embedding(
        node.get_content()
    )
    node.embedding = node_embedding

In [ ]:
# Check the total number of nodes
batch_size = len(nodes)
print(f"Batch size (number of nodes): {batch_size}") 

### Load Nodes into a Vector Store

the batch size of nodes (413) exceeds the maximum batch size allowed by ChromaDB (166). ChromaDB imposes a limit on the number of items you can add to a collection in a single batch. To resolve this issue, you can split your data into smaller batches and add them iteratively.

In [ ]:
for idx, node in enumerate(nodes):
    try:
        vector_store2.add([node])
    except Exception as e:
        print(f"Error with node {idx}: {e}")


In [ ]:
# Get the count of nodes in the collection
node_count = chroma_collection.count()
print(f"Number of nodes: {node_count}")

In [ ]:
for item in chroma_collection1.get():
    print(item)

In [ ]:
# Inspecting contents of the vector store
try:
    # Retrieve all stored items
    all_items = vector_store2.client.get()
    print(all_items)
    print("Contents of the vector store:")
    for item in all_items["documents"]:
        print(item)
except Exception as e:
    print(f"Error retrieving vector store contents: {e}")

### Querying

In [ ]:
query_str = "Can you tell me about what is OSS"

query_embedding = embed_model.get_query_embedding(query_str)

In [ ]:
# construct vector store query
from llama_index.core.vector_stores import VectorStoreQuery

query_mode = "default"
# query_mode = "sparse"
# query_mode = "hybrid"

vector_store_query = VectorStoreQuery(
    query_embedding=query_embedding, similarity_top_k=2, mode=query_mode
)

In [ ]:
# returns a VectorStoreQueryResult
query_result = vector_store2.query(vector_store_query)
print(query_result.nodes[0].get_content())

In [ ]:
from llama_index.core.schema import NodeWithScore
from typing import Optional

nodes_with_scores = []
for index, node in enumerate(query_result.nodes):
    score: Optional[float] = None
    if query_result.similarities is not None:
        score = query_result.similarities[index]
    nodes_with_scores.append(NodeWithScore(node=node, score=score))

In [ ]:
from llama_index.core import QueryBundle
from llama_index.core.retrievers import BaseRetriever
from typing import Any, List


class VectorDBRetriever(BaseRetriever):
    """Retriever over a postgres vector store."""

    def __init__(
        self,
        vector_store: ChromaVectorStore,
        embed_model: Any,
        query_mode: str = "default",
        similarity_top_k: int = 2,
    ) -> None:
        """Init params."""
        self._vector_store = vector_store2
        self._embed_model = embed_model
        self._query_mode = query_mode
        self._similarity_top_k = similarity_top_k
        super().__init__()

    def _retrieve(self, query_bundle: QueryBundle) -> List[NodeWithScore]:
        """Retrieve."""
        query_embedding = embed_model.get_query_embedding(
            query_bundle.query_str
        )
        vector_store_query = VectorStoreQuery(
            query_embedding=query_embedding,
            similarity_top_k=self._similarity_top_k,
            mode=self._query_mode,
        )
        query_result = vector_store2.query(vector_store_query)

        nodes_with_scores = []
        for index, node in enumerate(query_result.nodes):
            score: Optional[float] = None
            if query_result.similarities is not None:
                score = query_result.similarities[index]
            nodes_with_scores.append(NodeWithScore(node=node, score=score))

        return nodes_with_scores

In [ ]:
retriever = VectorDBRetriever(
    vector_store2, embed_model, query_mode="default", similarity_top_k=2
)

In [ ]:
from llama_index.core.query_engine import RetrieverQueryEngine

query_engine = RetrieverQueryEngine.from_args(retriever, llm=llm)

In [ ]:
query_str1 = "What is the definition of OSS?"

response1 = query_engine.query(query_str1)

In [ ]:
print(str(response1))

In [ ]:
print(response1.source_nodes[0].get_content())

In [ ]:
query_str2 = "What is early determination of distribution policy?"

response2 = query_engine.query(query_str2)


In [ ]:
print(response2.source_nodes[0].get_content())